# Phase 6 — Transfer Learning: Partial Fine-Tuning

Goal: unfreeze `layer4` -- the last of ResNet50's four residual blocks -- on top of
Phase 5's frozen-backbone starting point, and fine-tune it alongside a fresh
classifier head. `layer1`-`layer3` stay frozen (generic low-level features: edges,
textures), so only `layer4`'s higher-level features and the head adapt to this
dataset. Same fixed split and the exact Phase 3 harness (`src/training/`),
unmodified, as every other phase.

Uses a **discriminative learning rate**: the head trains at `lr=1e-3` (same as
Phase 5), `layer4` at `lr=1e-4` (10x lower) -- `layer4` already holds useful
pretrained features, and a large gradient from an untrained, randomly-initialized
head early in training would otherwise risk overwriting them before the head has
learned anything useful to backpropagate.

**Status:** authored and locally smoke-tested (imports, shapes, freeze/unfreeze
structure verified, a couple of training steps on a tiny subset, CPU-only here) --
the real 30-epoch run happens on Colab GPU, same as Phase 4/5. Takeaways at the
bottom get filled in with real numbers after that run.

In [ ]:
# Colab only: clone (or pull) the GitHub repo and cd into it. Safe to run locally
# too -- the import fails there and this cell is skipped. Repo is public, so a
# plain anonymous HTTPS clone works, no credentials needed.
try:
    import google.colab
    import os
    import shutil

    REPO_URL = "https://github.com/diljithG3/plant-disease-classification.git"
    REPO_DIR = "/content/plant-disease-classification"

    if os.path.isdir(os.path.join(REPO_DIR, ".git")):
        # A real repo from an earlier successful clone this session -- update it.
        _git_output = !git -C {REPO_DIR} pull 2>&1
    else:
        # No valid repo here yet -- possibly nothing, possibly a stray leftover
        # directory from an earlier failed attempt this session. Either way, wipe
        # it before cloning fresh: `git clone` refuses to write into a non-empty
        # directory, and re-running this cell after a failed first attempt would
        # otherwise silently take the `pull` branch above against a directory that
        # was never a real git repo, failing with "not a git repository".
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        _git_output = !git clone {REPO_URL} {REPO_DIR} 2>&1

    ok = os.path.isdir(os.path.join(REPO_DIR, ".git"))
    if not ok:
        print("git clone/pull failed:")
        print("\n".join(_git_output))
        raise RuntimeError("git clone/pull failed -- see output above.")

    # os.chdir, not the %cd magic -- %cd does not reliably substitute a Python
    # variable in {}, so it was cd-ing into a literal "{REPO_DIR}" path and
    # silently failing, leaving cwd at /content.
    os.chdir(REPO_DIR)
    print("Repo ready. cwd:", os.getcwd())
except ImportError:
    pass

In [ ]:
import sys, os

# Walk up from cwd to the project root (marked by configs/config.yaml) and put it on
# sys.path. Plain os.getcwd() is not reliable here -- it depends on how the kernel was
# launched (VS Code's Jupyter extension defaults to the notebook's own folder, not the
# project root).
_root = os.getcwd()
while not os.path.exists(os.path.join(_root, "configs", "config.yaml")):
    _parent = os.path.dirname(_root)
    if _parent == _root:
        raise RuntimeError("Could not find project root (configs/config.yaml) above " + os.getcwd())
    _root = _parent
sys.path.append(_root)

from pathlib import Path

import pandas as pd
import torch
import torch.nn as nn

from src.utils.config import load_config
from src.utils.seed import seed_everything
from src.utils.device import get_device
from src.data import splits, transforms as T, dataset as D
from src.data.fetch_data import fetch_dataset
from src.models.transfer import build_resnet50_partial_finetune
from src.training.trainer import train_model
from src.training.checkpoint import load_checkpoint
from src.training import engine
from src.training.metrics import plot_confusion_matrix

cfg = load_config()
seed_everything(cfg["seed"])
device = get_device(cfg["training"]["device"])
print("Environment:", cfg["env"], "| Device:", device)

fetch_dataset(cfg)  # no-op if already downloaded

## 1. Load Phase 2's fixed split

Reuses the split and `class_to_idx` cached by `02_data_pipeline.ipynb` unchanged --
same requirement as every previous phase: recomputing a split here would break
comparability with every other phase (`CLAUDE.md`).

In [ ]:
DATA_PATH = cfg["data"]["path"]
ARTIFACTS_DIR = cfg["data"]["artifacts_dir"]

splits_path = Path(ARTIFACTS_DIR) / "splits.csv"
class_to_idx_path = Path(ARTIFACTS_DIR) / "class_to_idx.json"

if not splits_path.exists() or not class_to_idx_path.exists():
    raise FileNotFoundError(
        f"{splits_path} or {class_to_idx_path} missing -- run notebooks/02_data_pipeline.ipynb first."
    )

splits_df = pd.read_csv(splits_path)
class_to_idx = splits.build_class_to_idx(DATA_PATH, cache_path=str(class_to_idx_path))
idx_to_class = {v: k for k, v in class_to_idx.items()}
num_classes = len(class_to_idx)

print(f"Loaded split: {len(splits_df)} images, {num_classes} classes")
splits_df["split"].value_counts()

## 2. Dataloaders -- full dataset

Same fixed split, same transforms as every other phase. Meant to run on Colab GPU --
local is CPU-only, fine for the cells above but not for a real training run.

In [ ]:
train_transform = T.get_train_transforms(cfg)
eval_transform = T.get_eval_transforms(cfg)

loaders = D.make_dataloaders(
    splits_df,
    class_to_idx,
    train_transform,
    eval_transform,
    batch_size=cfg["dataloader"]["batch_size"],
    num_workers=cfg["dataloader"]["num_workers"],
    data_path=DATA_PATH,
)

for split, loader in loaders.items():
    print(f"{split}: {len(loader.dataset)} images, {len(loader)} batches")

## 3. Model, loss, optimizer

`build_resnet50_partial_finetune` (`src/models/transfer.py`): same ResNet50 as
Phase 5, but `layer4` is left trainable alongside the fresh `fc` head --
`layer1`-`layer3` stay frozen.

Loss is unweighted `CrossEntropyLoss`, matching Phase 4 and Phase 5's choice, so any
accuracy/macro-F1 difference reflects the model/training change, not a different
imbalance-handling strategy. The optimizer uses two parameter groups with a
discriminative learning rate: `fc` at `1e-3` (same as Phase 5), `layer4` at `1e-4`
(10x lower) so its already-useful pretrained features aren't overwritten by large
early gradients from the still-untrained head.

In [ ]:
model = build_resnet50_partial_finetune(num_classes).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} total (layer4 + fc trainable, layer1-3 frozen)")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([
    {"params": model.fc.parameters(), "lr": 1e-3},
    {"params": model.layer4.parameters(), "lr": 1e-4},
])

## 4. Train

`max_epochs=30` and `early_stopping_patience: 7` on `macro_f1`
(`configs/config.yaml`) -- identical settings to every Phase 4/5 run, so only the
model and optimizer's learning-rate structure differ. Note: the logged `lr` column
in the training history below reflects only the first param group (`fc`)'s
learning rate, not `layer4`'s -- `src/training/trainer.py` logs
`optimizer.param_groups[0]["lr"]`.

In [ ]:
checkpoints_dir = Path(cfg["logging"]["checkpoints_dir"]) / "phase6_resnet50_partial_finetune"
resume_path = checkpoints_dir / "last.pt"
if resume_path.exists():
    print(f"Found checkpoint from an earlier attempt, resuming from {resume_path}")

history = train_model(
    model, loaders, criterion, optimizer, device,
    run_name="phase6_resnet50_partial_finetune", cfg=cfg, class_to_idx=class_to_idx, max_epochs=30,
    resume_from=resume_path if resume_path.exists() else None,
)
pd.DataFrame(history)

## 5. Verify checkpoint round-trip, then evaluate on the held-out test set

Same round-trip check as every previous phase: confirm `best.pt` reloads into a
**fresh** model instance and reproduces the metric it was saved with -- then use
that reloaded model for the real held-out test-set evaluation (not the val set used
during training/early stopping, which already influenced checkpoint selection).

In [ ]:
checkpoints_dir = Path(cfg["logging"]["checkpoints_dir"]) / "phase6_resnet50_partial_finetune"
best_path = checkpoints_dir / "best.pt"
assert best_path.exists()

fresh_model = build_resnet50_partial_finetune(num_classes).to(device)
payload = load_checkpoint(best_path, fresh_model, optimizer=None, map_location=device)
assert payload["class_to_idx"] == class_to_idx, "class_to_idx did not round-trip!"
print(f"Reloaded checkpoint from epoch {payload['epoch']}, best val {cfg['training']['checkpoint_metric']}={payload['best_metric']:.4f}")

test_metrics = engine.evaluate(fresh_model, loaders["test"], criterion, device, num_classes)
print(f"Test accuracy: {test_metrics['accuracy']:.4f} | Test macro_f1: {test_metrics['macro_f1']:.4f}")

## 6. Confusion matrix (test set)

In [ ]:
class_names = [idx_to_class[i] for i in range(num_classes)]
plot_confusion_matrix(test_metrics["confusion_matrix"], class_names)
import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()

## 7. Per-class recall vs. train-set frequency

Same check every previous phase ran: does the 36x class imbalance show up in which
classes this model struggles with, and does `Tomato___Early_blight` -- the chronic
weak class across Phase 4 and Phase 5 -- improve now that `layer4` can adapt
directly to this dataset's textures instead of relying on frozen, generic ImageNet
features?

In [ ]:
recall_series = pd.Series(
    test_metrics["per_class_recall"], index=[idx_to_class[i] for i in range(num_classes)]
).sort_values()

train_counts = splits_df[splits_df["split"] == "train"]["class"].value_counts()
comparison = pd.DataFrame({"train_count": train_counts, "test_recall": recall_series}).sort_values("train_count")

print("Lowest test recall (worst-performing classes):")
print(recall_series.head(10))
comparison

## Summary & implications for Phase 7

In [ ]:
print(f"Partial fine-tune test accuracy: {test_metrics['accuracy']:.4f}")
print(f"Partial fine-tune test macro_f1: {test_metrics['macro_f1']:.4f}")
print(f"Worst-recall class: {recall_series.index[0]} (recall={recall_series.iloc[0]:.3f}, train_count={int(train_counts.get(recall_series.index[0], 0))})")
print(f"Best-recall class: {recall_series.index[-1]} (recall={recall_series.iloc[-1]:.3f})")